# Kaleion · a first window
### Construct → inspect → count → transform → replay

This is a working exploration of the Python module. Every mathematical construction
is written in a cell you can change. Plotly supplies rotation, zoom, hover, playback,
and scrubbing; Kaleion supplies the values, relations, identities, provenance, and motion.

**Start:** install the notebook dependencies as described in [the setup guide](README.md),
select the **Kaleion** kernel, then **Restart Kernel and Run All Cells**. Run cells in
order after editing a construction; downstream snapshots do not update automatically.
There are no data downloads or services used by this notebook.

We will visit alternative arrangements, incidence counts, quotient/remainder tables,
3D drivers, the rectangular spiral, tensor operations, and reversible history.
The last section creates two embedded MP4 videos and standalone interactive HTML files.

**Reading the views:** teal = unfiltered arrangement, amber = relation satisfied,
grey = relation not satisfied, purple = membership changing along a presentation path.
Hover for exact labels and logical fields. A line is drawn only when we explicitly
choose storage order as a path. All observations here concern finite evaluated examples.

In [ ]:
from pathlib import Path
from math import gcd
import json
import sys

import numpy as np
import plotly
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Video, display

from kaleion import (
    Arrangement, Collection, Construction, F, Lens, Motion, Move,
    Sweep, Values, Workspace, choose, cos, graph, param, sin, vector,
)
from kaleion.viewers.plotly import animation_figure, snapshot_figure, transition_figure
from kaleion.viewers.video import write_mp4

# Rich MIME for JupyterLab/VS Code plus self-contained HTML for nbconvert.
# "notebook" embeds Plotly.js locally; "notebook_connected" would use a CDN.
pio.renderers.default = "plotly_mimetype+notebook"
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").exists() and (p / "src/kaleion").is_dir())
OUTPUT = ROOT / "build" / "notebooks"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kernel:", sys.executable)
print("Plotly:", plotly.__version__)
print("Generated files:", OUTPUT)

## 1 · The contents and their placement are independent

An integer sequence has values and a logical sequence index `s`. It need not have a
row or column. We can place the **same occurrences** on a line, in a snake, or in 3D.
Here `choose` controls the direction of alternate rows; `sin` and `cos` affect floating
coordinates, while the labels remain exact integers.

Try a different `WIDTH`, `LENGTH`, or `DIVISOR`. In 3D, drag to orbit and scroll to zoom.

In [ ]:
LENGTH, WIDTH, DIVISOR = 36, 6, 3
integers = Collection.sequence(LENGTH, start=1)
line = integers.arrange(F.s)
snake = integers.arrange(
    choose((F.s // WIDTH) % 2 == 0, F.s % WIDTH, WIDTH - 1 - F.s % WIDTH),
    -(F.s // WIDTH),
)
helix = integers.arrange(3 * cos(F.s * 0.45), 3 * sin(F.s * 0.45), F.s / 5)
multiples = Lens(F.value % DIVISOR == 0)

assert line.evaluate().ids == snake.evaluate().ids == helix.evaluate().ids
snake_plot = snapshot_figure(multiples(snake).evaluate(), connect=True,
                             title=f"One sequence, a snake · multiples of {DIVISOR}")
snake_plot.show()
helix_plot = snapshot_figure(multiples(helix).evaluate(), connect=True,
                             title="The same occurrences · a 3D placement")
helix_plot.show()

## 2 · A lens can read values, logical indices, or geometry

A lens is a Boolean rule over an arrangement's evaluated fields. Its spatial window
is half-open: lower bounds included, upper bounds excluded. Editing a placement can
change a geometric lens without changing a value-based relation. Moving points does
not silently rename their logical indices.

The following uses a parameter as a horizontal window position. The slider visits
exact cases; it does not calculate counts from interpolated pictures.

In [ ]:
window = multiples.window((param("left"), -6), (param("left") + 3, 1))
window_sweep = Sweep(window(snake), "left", range(-2, WIDTH))
window_plot = animation_figure(
    [window_sweep.at(i) for i in range(len(window_sweep.cases))],
    labels=[f"Window left = {left}" for left in window_sweep.cases],
    title="Move a spatial window across a value relation", duration=500,
)
window_plot.show()
# Replace the rule with Lens((F.s % 5 == 0) & (F.x < 3)) to mix index and geometry.

## 3 · An incidence becomes a new arrangement of integers

For `0 ≤ i < b` and `0 ≤ j < a`, place the value `a*i + b*j` at `(j, -i)`.
The lens selects values at least `a*b`. Then

\[
c_i = \#\{j : 0\le j<a,\; ai+bj\ge ab\}.
\]

`count(by=F.i)` **retains the key `i`** and reduces over the other index. A zero count
still has an occurrence and a key. The profile below is constructed from those
cardinalities through the public API, not computed by the plotting code.

Change `PARAMETERS` to another pair of positive coprime integers to explore both this
region and the remainder-table construction in the next section. Coprimality is a
requirement of that particular row-ordering example, not of `count` or `roll`.

These cardinalities are **exactly** $\lfloor ai/b\rfloor$: putting $y=a-j$ turns
$ai+bj\ge ab$ into $1\le y\le ai/b$. The $i=0$ group supplies the leading zero.
This count identity holds even without coprimality. See
[Two incidences fill a rectangle](02_floor_sum_proof.ipynb) for the complementary
incidence, an animated construction, and a general proof of the reciprocal floor-sum identity.


In [ ]:
PARAMETERS = {"a": 11, "b": 7}
assert PARAMETERS["a"] > 1 and PARAMETERS["b"] > 1
assert gcd(PARAMETERS["a"], PARAMETERS["b"]) == 1, "Use coprime a,b for the remainder example"
a, b = param("a"), param("b")
region = Collection.grid(b, a, values=a * F.i + b * F.j).arrange(F.j, -F.i)
incidence = region.where(F.value >= a * b)
counts = incidence.count(by=F.i)
profile = counts.arrange(F.key, F.value)

incidence_snapshot = incidence.evaluate(**PARAMETERS)
count_snapshot = profile.evaluate(**PARAMETERS)
print("Counts:", count_snapshot.values.tolist())
print("Total incidence:", incidence_snapshot.cardinality)
assert sum(count_snapshot.values) == incidence_snapshot.cardinality
assert count_snapshot.values[0] == 0

quotient_plot = snapshot_figure(incidence_snapshot, title="Quotient region · values ≥ a b")
quotient_plot.show()
profile_plot = snapshot_figure(count_snapshot, connect=True,
                               title="A new arrangement · x = retained key, y = cardinality")
profile_plot.show()

In [ ]:
# Inspect the contributors to one count. These IDs refer to the original universe.
INSPECT_KEY = min(3, PARAMETERS["b"] - 1)
contributors = count_snapshot.contributor_ids(INSPECT_KEY)
original_values = dict(zip(incidence_snapshot.source.ids, incidence_snapshot.source.values))
print("Formula:", count_snapshot.metadata["formula"])
print(f"Key {INSPECT_KEY} contributors:", [int(original_values[oid]) for oid in contributors])
print("Key zero contributors:", count_snapshot.contributor_ids(0))

# Counts are immediately usable as inputs to another lens and reduction.
positive_even_counts = profile.where((F.value > 0) & (F.value % 2 == 0))
print("Positive even cardinalities:", positive_even_counts.select().evaluate(**PARAMETERS).values.tolist())
print("How many such groups:", positive_even_counts.count().evaluate(**PARAMETERS).values.tolist())

## 4 · Gather rows, then let counts drive cyclic shifts

Now put `i + b*j` in the slots of a remainder table. The multiples of `a`, read in
increasing value order, tell us which original rows to gather. The count at key `i`
then drives a cyclic shift along `j`. Binding is by declared keys, not storage order.

Watch the labels move to new slots and then return on undo. The arc is a presentation
path; this first viewer does not yet draw the special cut-and-reassemble wrap.

In [ ]:
remainder = Collection.grid(b, a, values=F.i + b * F.j).arrange(F.j, -F.i)
row_addresses = remainder.where(F.value % a == 0).select().order_by(F.value).with_values(F.i)
ordered = remainder.items.gather(row_addresses, axis="i").arrange(F.j, -F.i)
rolled = ordered.roll(axis="j", shift=-profile.bind(on=F.i, key=F.key, read=F.value))

rolled_snapshot = rolled.evaluate(**PARAMETERS)
first_column = rolled_snapshot.values.reshape(rolled_snapshot.shape)[:, 0].tolist()
print("Gather addresses:", row_addresses.evaluate(**PARAMETERS).values.tolist())
print("Gather + Roll first column:", first_column)
assert first_column == [PARAMETERS["a"] * i for i in range(PARAMETERS["b"])]

snapshot_figure(remainder.where(F.value % a == 0).evaluate(**PARAMETERS),
                title="Before gathering · multiples of a in the remainder table").show()
table_workspace = Workspace({"table": ordered}, PARAMETERS)
roll_forward = table_workspace.set("table", rolled, motion=Motion.arc(height=0.6, axis=1))
roll_undo = table_workspace.undo()
times = np.linspace(0, 1, 49)
roll_frames = [step.frame("table", float(t))
               for step in (roll_forward, roll_undo) for t in times]
roll_labels = [f"{direction} · {t:.0%}" for direction in ("Roll", "Undo") for t in times]
roll_plot = animation_figure(roll_frames, labels=roll_labels,
                             title="Cardinalities drive a roll · then undo the edit", duration=45)
roll_plot.show()

## 5 · The same driver can act on a scattered 3D arrangement

There is no row structure in this cloud. Each target value requests a key using
`value % b`. The bound cardinality can change **placement** with `Move`, or **contents**
with `Values`. The inputs are the same; the chosen operation declares what changes.

The 3D animation samples the recorded arc, including undo. You can orbit while paused.
Kaleion restores exact state on undo; it does not numerically integrate a reverse path.

In [ ]:
cloud = Arrangement.points(range(9), [(i % 3, (i * i) % 5, i % 2) for i in range(9)])
amount = profile.bind(on=F.value % b, key=F.key, read=F.value)
moved_cloud = cloud | Move(vector(amount, 0, -amount))
changed_cloud = cloud | Values(F.value + amount)
print("Changed contents:", changed_cloud.evaluate(**PARAMETERS).values.tolist())
np.testing.assert_array_equal(cloud.evaluate().positions,
                              changed_cloud.evaluate(**PARAMETERS).positions)

workspace = Workspace({"cloud": cloud, "counts": profile}, PARAMETERS)
forward = workspace.set("cloud", moved_cloud, motion=Motion.arc(height=2, axis=1, dimension=3))
backward = workspace.undo()
cloud_frames = [step.frame("cloud", float(t)) for step in (forward, backward) for t in times]
cloud_plot = animation_figure(cloud_frames, labels=[
    f"{direction} · {t:.0%}" for direction in ("Move", "Undo") for t in times
], title="A keyed driver in 3D · move and undo", duration=45)
cloud_plot.show()

for t in (0.0, 0.25, 0.5, 0.75, 1.0):
    np.testing.assert_allclose(forward.frame("cloud", 1 - t).positions,
                               backward.frame("cloud", t).positions, rtol=0, atol=1e-12)
print("Undo follows the original captured path at 1 − t.")

## 6 · The rectangular spiral: inspect structure, not primality

The spiral starts at **1**, with no empty zero slot. Its initial straight run has
length `n`. The constructor emits a `cycle_end` structural role at each completed
growth cycle; the lens reads that role. It contains no test for divisibility or primes.

The first playback visits exact integer `n` cases. The second shows motion between two
cases; its intermediate positions are presentation samples.

In [ ]:
LIMIT = 36
spiral = Arrangement.spiral(LIMIT, initial=param("n"))
structural_hits = spiral.where(F.cycle_end)
spiral_sweep = Sweep(structural_hits, "n", range(1, LIMIT // 2))
spiral_cases = [spiral_sweep.at(i) for i in range(len(spiral_sweep.cases))]
spiral_labels = [f"n = {n}" for n in spiral_sweep.cases]
spiral_plot = animation_figure(spiral_cases, labels=spiral_labels,
                               title="Rectangular spiral · exact cases, structural cycle ends",
                               connect=True, duration=550)
spiral_plot.show()

In [ ]:
unroll = spiral_sweep.transition(0, 1)
unroll_plot = transition_figure(unroll, "result", steps=41, connect=True,
                                title="Between n = 1 and n = 2 · presentation motion")
unroll_plot.show()

# Retain includes every exact case through this index, even if playback skipped cases.
seen = spiral_sweep.retain(len(spiral_sweep.cases) - 1)
seen_values = sorted(hit["value"] for hit in seen)
print("Retained structural hits:", seen_values)
# Independent finite check, performed AFTER the structural observation.
composites = [n for n in range(2, LIMIT + 1)
              if any(n % d == 0 for d in range(2, int(n ** 0.5) + 1))]
assert seen_values == composites
print(f"Through {LIMIT}: {len(seen_values)} hits, exactly the composites in this finite check.")
print("The remaining values include 1; 1 is neither prime nor composite.")

Retention is an observation over the sweep, separate from the changing arrangement.
Below is a custom Plotly view of that observation: the bottom row lists **every**
retained value; the top row lists values not retained. This display does not add a new
symbolic construction or claim a proof.

In [ ]:
retained = set(seen_values)
labels = list(range(1, LIMIT + 1))
retention_plot = go.Figure(go.Scatter(
    x=labels, y=[0 if n in retained else 1 for n in labels],
    mode="markers+text", text=[str(n) for n in labels], textposition="top center",
    marker=dict(size=10, color=["#f5bd59" if n in retained else "#48cdb4" for n in labels]),
    hovertemplate="Value %{text}<extra></extra>",
))
retention_plot.update_layout(
    template="plotly_dark", height=280, paper_bgcolor="#101827", plot_bgcolor="#101827",
    title="History of structural hits · every exact case is included",
    xaxis=dict(title="Integer value", dtick=2),
    yaxis=dict(tickvals=[0, 1], ticktext=["Retained", "Not retained"], range=[-0.5, 1.7]),
    margin=dict(l=110, r=25, t=65, b=45),
)
retention_plot.show()

## 7 · A derived count can even become a constructor argument

A per-item driver and a single constructor argument are different contracts.
`.scalar()` explicitly requires one selected integer. Here the count at key `i=3`
(with a positive-count fallback when that key is zero or unavailable) sets the spiral's initial run.
The construction graph retains this dependency.

In [ ]:
# A straight-run length must be positive; some other parameter pairs have a zero at key 3.
SEED_KEY = INSPECT_KEY if count_snapshot.values[INSPECT_KEY] > 0 else PARAMETERS["b"] - 1
seed = counts.where(F.key == SEED_KEY).select().scalar()
spiral_tool = Construction(Arrangement.spiral(LIMIT, initial=param("n")), ("n",))
driven_spiral = spiral_tool(n=seed)
print("Derived initial run:", counts.where(F.key == SEED_KEY).select().evaluate(**PARAMETERS).values.tolist())
print("Cycle endpoints:", driven_spiral.where(F.cycle_end).select().evaluate(**PARAMETERS).values.tolist())
snapshot_figure(driven_spiral.where(F.cycle_end).evaluate(**PARAMETERS), connect=True,
                title="A spiral whose constructor reads an incidence count").show()

## 8 · Gather, substitution, tile, pad, and a Young diagram

Gather chooses **occurrences by address**, and may repeat or omit inputs. Lookup
substitutes **values from a table**. Neither needs a matrix. Tiling creates repeated
occurrences; padding creates actual new integer contents. These operations return
collections whose placement is declared separately.

A Young diagram declares cells of a partition. A filling is not automatically a
standard or semistandard tableau; those would need additional predicates.

In [ ]:
base = Collection.literal([0, 1, 2, 3])
gathered = base.gather([3, 0, 3, 1])
substituted = base.lookup([10, 30, 20, 40], address=F.value)
tiled_padded = base.tile(2).pad(1, 1, value=0, attribute_fill={"s": -1})
for name, construction in [("Gather", gathered), ("Lookup", substituted), ("Tile + pad", tiled_padded)]:
    print(name, construction.evaluate().values.tolist())
snapshot_figure(tiled_padded.arrange(F.index).evaluate(), title="Tile and pad · distinct occurrences, repeated labels").show()

young = Collection.young([5, 3, 2], values=F.index + 1).arrange(F.j, -F.i)
snapshot_figure(young.where(F.j <= F.i).evaluate(), title="Young diagram · a lens on cell indices").show()

# A 3-axis logical domain can likewise be placed in 3D or projected into 2D.
volume = Collection.grid(4, 4, 4, values=F.i + 4 * F.j + 16 * F.k).arrange(F.i, F.j, F.k)
plane = volume.where(F.i + F.j + F.k == 4)
snapshot_figure(plane.evaluate(), title="Three logical axes · the incidence i + j + k = 4",
                show_values=False).show()
print("Plane count by k:", plane.count(by=F.k).evaluate().values.tolist())

## 9 · Save an observation and replay without evaluating it again

The workspace stores exact captured states and the recorded path. A saved observation
survives later parameter changes. Renderers only consume snapshots or frames;
they never supply interpolated values to `count`, bindings, or subsequent mathematics.

In [ ]:
workspace.redo()
observation = workspace.capture("Quotient-region counts drive displacement of a scattered 3D sequence.")
workspace.set_parameters(a=PARAMETERS["a"] + 1)
print("Edited counts:", workspace.state.results["counts"].values.tolist())
workspace.restore(observation)
np.testing.assert_array_equal(workspace.state.results["counts"].values, count_snapshot.values)

workspace_path = OUTPUT / "investigation.json"
workspace_path.write_text(workspace.to_json(), encoding="utf-8")
reopened = Workspace.from_json(workspace_path.read_text(encoding="utf-8"))
np.testing.assert_array_equal(reopened.state.results["cloud"].positions,
                              workspace.state.results["cloud"].positions)
definitions = graph({"counts": counts.node, "moved": moved_cloud.node,
                     "rolled": rolled.node, "driven_spiral": driven_spiral.node})
(OUTPUT / "construction-graph.json").write_text(json.dumps(definitions, indent=2), encoding="utf-8")
print("Saved definitions, captured results, provenance, and history.")

## 10 · Playable videos and standalone interactive exports

Plotly's animations retain hover, scrubbing, zoom, and 3D rotation. An MP4 is a raster
movie with standard video controls. `write_mp4` renders the **same sampled results**
using Pillow and an FFmpeg encoder; it does not record or require a browser.
The initial MP4 adapter supports 1D/2D; use Plotly's interactive HTML for 3D.

These cells run as part of **Run All**. Both movies are embedded in the executed
notebook. Generated files live in the ignored `build/notebooks/` directory.

In [ ]:
roll_video = write_mp4(roll_frames, OUTPUT / "roll-and-undo.mp4",
                        labels=roll_labels, title="Kaleion · cardinalities drive motion", fps=24)
display(Video(filename=str(roll_video), embed=True, width=800, html_attributes="controls loop"))

# Hold each exact case for half a second. No new mathematical cases are invented.
spiral_video = write_mp4(
    [sample for sample in spiral_cases for _ in range(12)],
    OUTPUT / "spiral-cases.mp4",
    labels=[label for label in spiral_labels for _ in range(12)],
    title="Kaleion · structural discovery", fps=24, connect=True,
)
display(Video(filename=str(spiral_video), embed=True, width=800, html_attributes="controls loop"))

In [ ]:
# Each file embeds Plotly.js and works offline after installation.
for name, figure in [("spiral", spiral_plot), ("roll-and-undo", roll_plot), ("cloud-3d", cloud_plot)]:
    figure.write_html(OUTPUT / f"{name}.html", include_plotlyjs=True, auto_play=False,
                       config={"responsive": True, "displaylogo": False})
print("Open these files in your browser:")
for name in ("spiral.html", "roll-and-undo.html", "cloud-3d.html",
             "roll-and-undo.mp4", "spiral-cases.mp4",
             "investigation.json", "construction-graph.json"):
    path = OUTPUT / name
    print(f"  {path.name:28} {path.stat().st_size / 1024:9.1f} KiB")

## Where to experiment next

1. Change the quotient-region predicate, then inspect how its cardinalities alter both
   the remainder table and the 3D displacement. The first-column equality is specific
   to the original construction; an assertion may correctly stop passing after an edit.
2. Replace the snake placement with your own symbolic coordinates, then apply the same
   lens. Which patterns belong to the values, and which depend on placement?
3. Change `count(by=F.i)` to another grouping and inspect its keys before reusing it.
   Missing or ambiguous driver keys are explicit errors, never silent zero displacements.
4. Give an edit a `Motion.custom(...)` path. Compare undo frames against the forward path
   at `1-t`. A lossy mathematical edit can still have exact state restoration.
5. Construct a new 3D incidence and reduce it along retained axes. The displayed geometry
   and the logical tensor domain are separate choices.

The most useful feedback is a small construction that was awkward or impossible to
express. That will tell us whether we need a primitive, clearer syntax, or only a better view.

**Adapter references:** [Plotly animation frames](https://plotly.com/python/animations/),
[offline interactive HTML](https://plotly.com/python/interactive-html-export/),
[display renderers](https://plotly.com/python/renderers/), and
[imageio-ffmpeg encoding](https://github.com/imageio/imageio-ffmpeg).